# RLSF — scoring the three arms on val

---
## 1 — Preconditions

In [2]:
%cd /home/prnamhr/projects/Style-Aware-MT
import json
import subprocess
import sys
from pathlib import Path

import yaml

from src.eval.stylometrics_ci import MAIN_CONDITIONS

PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
SPLIT = 'val'

ARMS = {'RL-Metric': 'w3_0.0', 'RLSF-Judge': 'w3_2.0', 'RLSF-Judge-High': 'w3_6.0'}
ARM_CONDS = [f'rlsf_{cell}' for cell in ARMS.values()]
ALL_CONDS = MAIN_CONDITIONS + ARM_CONDS
print(' '.join(ALL_CONDS))

/home/prnamhr/projects/Style-Aware-MT
zeroshot random_fewshot knn_fewshot afsp_margin afsp_full peft commercial_haiku rlsf_w3_0.0 rlsf_w3_2.0 rlsf_w3_6.0


In [3]:
VAL = [json.loads(x) for x in open(f'data/splits/{SPLIT}.jsonl') if x.strip()]

# The hypothesis records do not carry the adapter, so which checkpoint wrote them is only
# recoverable from the config Phase B froze. Read it here so the provenance is on the page.
for cell, cond in zip(ARMS.values(), ARM_CONDS):
    path = Path('outputs') / f'{cond}_{SPLIT}.jsonl'
    rows = [json.loads(x) for x in open(path) if x.strip()]
    assert len(rows) == len(VAL), f'{path}: {len(rows)} rows, expected {len(VAL)}'
    assert all(a['input'] == b['input'] for a, b in zip(rows, VAL)), f'{path}: misalignment'
    assert not [r for r in rows if r.get('error') or not r['prediction'].strip()]

    eval_cfg = yaml.safe_load(Path(f'configs/rlsf_eval_{cell}.yaml').read_text())
    sel = json.loads(Path(f'results/rlsf_select_{cell}.json').read_text())
    assert eval_cfg['generator']['adapter_path'] == sel['selected_path'], cell
    print(f"{cond:14s} {len(rows)} segments, {sel['selected']} @ {sel['selected_path']}")

# Every paid call below lands on val. The test split stays sealed.
assert not any(Path('outputs').glob('*_test.jsonl')), 'a test-split output exists'

rlsf_w3_0.0    1323 segments, step200 @ models/rlsf_grpo_w3_0.0/checkpoint-200
rlsf_w3_2.0    1323 segments, step200 @ models/rlsf_grpo_w3_2.0/checkpoint-200
rlsf_w3_6.0    1323 segments, step100 @ models/rlsf_grpo_w3_6.0/checkpoint-100


In [4]:
!git rev-parse --short HEAD
!git status --short outputs results

b1b9b49


In [7]:
# %pip installs into the kernel; !pip may not.
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import numpy
import transformers

# COMET gets its own interpreter. Installing requirements-comet.txt into the kernel downgrades
# transformers and numpy under the generator, which then runs on a stack nothing else uses.
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)
subprocess.run([COMET_PY, '-c', 'import comet; print("comet ok")'], check=True)

print(f'kernel transformers {transformers.__version__}, numpy {numpy.__version__}')
assert transformers.__version__.startswith('5.'), "COMET's pins landed in the kernel"
assert numpy.__version__.startswith('2.'), "COMET's pins landed in the kernel"

/home/prnamhr/projects/Style-Aware-MT/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


comet ok
kernel transformers 5.12.1, numpy 2.4.1


---
## 2 — Surface overlap and the register proxy

In [5]:
!{PY} manage.py eval --conditions {' '.join(ALL_CONDS)} --split {SPLIT}

condition         n     BLEU   chrF   marker_rate  ref_marker_rate
------------------------------------------------------------------
zeroshot          1323  10.27  36.42  1.4          0.79           
random_fewshot    1323  11.64  37.52  1.14         0.79           
knn_fewshot       1323  13.99  39.82  0.9          0.79           
afsp_margin       1323  13.69  39.68  0.96         0.79           
afsp_full         1323  14.52  39.99  0.98         0.79           
peft              1323  16.9   41.58  0.92         0.79           
commercial_haiku  1323  18.06  45.24  1.11         0.79           
rlsf_w3_0.0       1323  17.01  41.85  0.93         0.79           
rlsf_w3_2.0       1323  17.0   42.04  0.99         0.79           
rlsf_w3_6.0       1323  17.13  42.08  0.95         0.79           


---
## 3 — COMET

In [9]:
subprocess.run(
    [COMET_PY, 'manage.py', 'comet', '--conditions', *ARM_CONDS, '--split', SPLIT],
    check=True,
)

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1280.86it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/prnamhr/projects/Style-Aware-MT/.venv-comet/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and ver

rlsf_w3_0.0      COMET 0.7007  (n=1323)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/prnamhr/projects/Style-Aware-MT/.venv-comet/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


rlsf_w3_2.0      COMET 0.7007  (n=1323)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/prnamhr/projects/Style-Aware-MT/.venv-comet/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


rlsf_w3_6.0      COMET 0.6993  (n=1323)
preserved 7 condition(s) not scored here: afsp_full, afsp_margin, commercial_haiku, knn_fewshot, peft, random_fewshot, zeroshot
Wrote results/comet_val.json


CompletedProcess(args=['.venv-comet/bin/python', 'manage.py', 'comet', '--conditions', 'rlsf_w3_0.0', 'rlsf_w3_2.0', 'rlsf_w3_6.0', '--split', 'val'], returncode=0)

---
## 4 — The two raters

In [10]:
N_CALLS = len(ARM_CONDS) * len(VAL)

u = json.loads(Path('results/judge_gpt_val_usage.json').read_text())['cumulative']
rate_b = u['cost_usd'] / u['calls']
# Estimated, not measured: no usage artefact exists for the Phi_A pass (docs/budget.md).
RATE_A_EST = 6.187e-4
AUTHORISED = 5.74

print(f"Phi_B gpt-5.6-terra   {N_CALLS} calls x ${rate_b:.4e} = ${N_CALLS * rate_b:.2f}  "
      f"(batch, 50% discount; rate measured over {u['calls']} calls)")
print(f"Phi_A claude-haiku-4-5 {N_CALLS} calls x ${RATE_A_EST:.4e} = ${N_CALLS * RATE_A_EST:.2f}  "
      f"(synchronous; rate ESTIMATED)")
print(f"\ntotal ~${N_CALLS * (rate_b + RATE_A_EST):.2f} against ${AUTHORISED:.2f} authorised")
assert N_CALLS * (rate_b + RATE_A_EST) <= AUTHORISED, 'over the authorised band; do not submit'

Phi_B gpt-5.6-terra   3969 calls x $6.6054e-04 = $2.62  (batch, 50% discount; rate measured over 9261 calls)
Phi_A claude-haiku-4-5 3969 calls x $6.1870e-04 = $2.46  (synchronous; rate ESTIMATED)

total ~$5.08 against $5.74 authorised


In [11]:
# The rubric both raters read is the evaluation one, distinct from the reward rubric the arms
# were trained against. Scoring on the training rubric would be circular.
import hashlib

for path in ('configs/judge_eval.yaml', 'configs/judge_eval_gpt.yaml'):
    c = yaml.safe_load(Path(path).read_text())
    assert c['template_file'] == 'prompts/judge_eval.txt', c['template_file']
    print(f"{path:30s} {c['judge']['model']:20s} tag={c.get('tag') or '(none)'}")

frozen = json.loads(Path('prompts/hashes.json').read_text())['templates']
digest = hashlib.sha256(Path('prompts/judge_eval.txt').read_bytes()).hexdigest()
assert digest == frozen['judge_eval.txt']['sha256'], 'the evaluation rubric has drifted'
print(f'\nrubric verified {digest[:16]}')

configs/judge_eval.yaml        claude-haiku-4-5     tag=(none)
configs/judge_eval_gpt.yaml    gpt-5.6-terra        tag=gpt

rubric verified ffd6dad41acb0512


In [12]:
# Phi_B. The Batch API is submit-then-poll; the batch id is persisted before the first poll, so
# an interrupted cell resumes the same batch instead of paying for a second one.
!{PY} manage.py judge_batch --conditions {' '.join(ARM_CONDS)} --split {SPLIT} \
    --config configs/judge_eval_gpt.yaml

judge gpt-5.6-terra [batch]  tag=gpt  template=prompts/judge_eval.txt [ffd6dad41acb0512]
Judging 1323 segments for rlsf_w3_0.0 with gpt-5.6-terra ...
  submitting 1323 requests ...
  submitted batch batch_6a80b2e8e88c8190b0fee747283c47e0
  [validating] 0/0 completed
  [validating] 0/0 completed
  [validating] 0/0 completed
  [in_progress] 0/1323 completed
  [in_progress] 379/1323 completed
  [in_progress] 723/1323 completed
  [in_progress] 1095/1323 completed
  [finalizing] 1323/1323 completed
  [finalizing] 1323/1323 completed
  [completed] 1323/1323 completed
  batch completed: wrote 1323 segment(s), 13 unscored
  rlsf_w3_0.0      Φ 3.598  (coverage 99%)
Judging 1323 segments for rlsf_w3_2.0 with gpt-5.6-terra ...
  submitting 1323 requests ...
  submitted batch batch_6a80b4026090819099c409a9bf563401
  [validating] 0/0 completed
  [validating] 0/0 completed
  [validating] 0/0 completed
  [in_progress] 0/1323 completed
  [in_progress] 611/1323 completed
  [in_progress] 1061/1323 compl

In [13]:
# Phi_A, synchronous: judge_batch rejects a non-OpenAI provider.
!{PY} manage.py judge --conditions {' '.join(ARM_CONDS)} --split {SPLIT} \
    --config configs/judge_eval.yaml

judge claude-haiku-4-5  tag=(none)  template=prompts/judge_eval.txt [ffd6dad41acb0512]
Judging 1323 segments for rlsf_w3_0.0 with claude-haiku-4-5 ...
  rlsf_w3_0.0      Φ 2.769  (coverage 100%)
Judging 1323 segments for rlsf_w3_2.0 with claude-haiku-4-5 ...
  rlsf_w3_2.0      Φ 2.791  (coverage 100%)
Judging 1323 segments for rlsf_w3_6.0 with claude-haiku-4-5 ...
  rlsf_w3_6.0      Φ 2.742  (coverage 100%)
preserved 7 condition(s) not re-scored: afsp_full, afsp_margin, commercial_haiku, knn_fewshot, peft, random_fewshot, zeroshot
Wrote results/judge_val.json
Judge usage: {'calls': 3969, 'prompt_tokens': 1881647, 'completion_tokens': 434600, 'cost_usd': 4.0546}
Wrote results/judge_val_usage.json  (cumulative $4.05)


In [14]:
for tag, path in (('Phi_A', 'results/judge_val_usage.json'),
                  ('Phi_B', 'results/judge_gpt_val_usage.json')):
    if not Path(path).exists():
        print(f'{tag}: no usage artefact ({path})')
        continue
    d = json.loads(Path(path).read_text())
    s, c = d['session'], d['cumulative']
    print(f"{tag} {d['model']:20s} this pass {s['calls']:5d} calls ${s['cost_usd']:.4f}  "
          f"| cumulative {c['calls']:6d} calls ${c['cost_usd']:.4f}")

Phi_A claude-haiku-4-5     this pass  3969 calls $4.0546  | cumulative   3969 calls $4.0546
Phi_B gpt-5.6-terra        this pass  3969 calls $2.6047  | cumulative  13230 calls $8.7220


---
## 5 — Judge intervals and cross-rater agreement

In [15]:
!{PY} manage.py judge_ci --conditions {' '.join(ALL_CONDS)} --split {SPLIT}
!{PY} manage.py judge_ci --conditions {' '.join(ALL_CONDS)} --split {SPLIT} --tag gpt


Judge register fidelity Phi by condition  (split=val, n=1323 segments, resamples=10000, seed=42)
judge: claude-haiku-4-5  [tag (none)]
Phi = mean 1-5 rubric rating against the authorized reference; higher is better.

rank  condition         class      n     Phi     ci95              sd     P(this rank)  modal rank  mean rank
-------------------------------------------------------------------------------------------------------------
1     commercial_haiku  reference  1323  3.3333  [3.2880, 3.3787]  0.843  1.000         1 (1.000)   1.00     
2     afsp_full         study      1323  2.7906  [2.7385, 2.8420]  0.948  0.465         2 (0.465)   2.99     
3     rlsf_w3_2.0       study      1323  2.7906  [2.7369, 2.8458]  1.015  0.337         2 (0.460)   2.82     
4     rlsf_w3_0.0       study      1323  2.7687  [2.7135, 2.8246]  1.034  0.311         4 (0.311)   4.44     
5     afsp_margin       study      1323  2.7627  [2.7135, 2.8133]  0.915  0.233         5 (0.233)   5.02     
6     knn_fe

In [16]:
!{PY} manage.py judge_agreement --conditions {' '.join(ALL_CONDS)} --split {SPLIT} --tag_b gpt


Judge-judge agreement  (split=val, resamples=10000, seed=42, 95% percentile CIs)
  judge A: claude-haiku-4-5  [tag (none)]
  judge B: gpt-5.6-terra  [tag gpt]
  same frozen rubric verified by digest: True

Coverage (segments parsed by each rater)
condition         n_total  n_a   n_b   n_both
---------------------------------------------
zeroshot          1323     1323  1300  1300  
random_fewshot    1323     1323  1297  1297  
knn_fewshot       1323     1322  1308  1307  
afsp_margin       1323     1323  1303  1303  
afsp_full         1323     1323  1310  1310  
peft              1323     1323  1306  1306  
commercial_haiku  1323     1323  1308  1308  
rlsf_w3_0.0       1323     1323  1310  1310  
rlsf_w3_2.0       1323     1323  1307  1307  
rlsf_w3_6.0       1323     1323  1310  1310  

Rater agreement -- study_only
condition         n      Phi_A  Phi_B  A-B     ci95              qwk     qwk_ci            rho     exact  adj  
---------------------------------------------------------

---
## 6 — Stylometrics

In [ ]:
# targets-split train puts the target register itself in the table, as the row every
# condition's distance is measured against.
!{PY} manage.py stylometrics --conditions {' '.join(ALL_CONDS)} --split {SPLIT} \
    --targets-split train

label             n      lex_density  lex_density_sd  ttr     ttr_sd  root_ttr  root_ttr_sd  sent_len_mean  sent_len_mean_sd  sent_len_var  sent_len_var_sd  marker_rate  marker_rate_sd  stylo_dist
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
target:train      10860  0.4344       0.1101          0.854   0.1085  4.0437    1.0426       24.7388        16.6125           6.6894        58.4011          0.0327       0.0567          0.0       
zeroshot          1323   0.4025       0.0999          0.8434  0.1222  3.858     0.9711       23.7585        19.5748           4.5813        32.7304          0.0637       0.0817          0.6518    
random_fewshot    1323   0.4103       0.1116          0.8444  0.1175  3.8491    0.9751       23.5958        15.5458           5.5118        47.8312          0.0534       0.0753          0.4736    
knn_fewshot    

In [18]:
!{PY} manage.py stylometrics_ci --conditions {' '.join(ALL_CONDS)} --split {SPLIT}


Register fit of the main conditions  (split=val, n=1323 segments, resamples=2000, seed=42)
stylo_dist = standardized distance to the target-register centroid; lower is better.

rank  condition         stylo_dist  ci95              P(this rank)  modal rank  mean rank
-----------------------------------------------------------------------------------------
1     peft              0.2886      [0.2410, 0.3466]  0.725         1 (0.725)   1.34     
2     rlsf_w3_0.0       0.2960      [0.2460, 0.3541]  0.451         2 (0.451)   2.16     
3     rlsf_w3_6.0       0.2990      [0.2502, 0.3585]  0.578         3 (0.578)   2.51     
4     rlsf_w3_2.0       0.3279      [0.2756, 0.3900]  0.932         4 (0.932)   4.08     
5     afsp_full         0.3698      [0.3238, 0.4214]  0.789         5 (0.789)   5.11     
6     afsp_margin       0.3910      [0.3471, 0.4444]  0.622         6 (0.622)   6.16     
7     knn_fewshot       0.4005      [0.3568, 0.4532]  0.691         7 (0.691)   6.64     
8     random

---
## 7 — Held-out register distance against judge score

In [19]:
import numpy as np

from src.eval.stylometrics import (
    HELDOUT_FEATURES,
    REWARD_FEATURES,
    aggregate,
    bootstrap_draws,
    distance_to_centroid,
    feature_vector,
    signed_z,
    subcentroid,
)

centroid = json.loads(Path('results/stylometrics_centroid_split.json').read_text())
held = subcentroid(centroid, HELDOUT_FEATURES)
reward = subcentroid(centroid, REWARD_FEATURES)

phi = {t: json.loads(Path(p).read_text())
       for t, p in (('A', 'results/judge_val.json'), ('B', 'results/judge_gpt_val.json'))}


def read(cond):
    rows = [json.loads(x) for x in open(f'outputs/{cond}_{SPLIT}.jsonl') if x.strip()]
    return [r['prediction'] for r in rows]


REPORT = ['peft', *ARM_CONDS]
table = {}
for cond in REPORT:
    preds = read(cond)
    agg = aggregate(preds)
    matrix = np.asarray([feature_vector(t) for t in preds if t.strip()], dtype=float)
    dists, _ = bootstrap_draws(matrix, held, n_resamples=2000, seed=42)
    table[cond] = {
        'dist_heldout': distance_to_centroid(agg['mean'], held),
        'ci': (float(np.percentile(dists, 2.5)), float(np.percentile(dists, 97.5))),
        'dist_reward': distance_to_centroid(agg['mean'], reward),
        'z': signed_z(agg['mean'], centroid),
        'phi_a': phi['A'].get(cond, {}).get('mean'),
        'phi_b': phi['B'].get(cond, {}).get('mean'),
    }

print(f"{'condition':16s} {'d_heldout':>10s} {'95% CI':>18s} {'d_reward':>9s} "
      f"{'Phi_A':>7s} {'Phi_B':>7s}")
for cond, r in table.items():
    a = f"{r['phi_a']:.3f}" if r['phi_a'] is not None else '  n/a'
    b = f"{r['phi_b']:.3f}" if r['phi_b'] is not None else '  n/a'
    print(f"{cond:16s} {r['dist_heldout']:10.4f} "
          f"[{r['ci'][0]:7.4f},{r['ci'][1]:7.4f}] {r['dist_reward']:9.4f} {a:>7s} {b:>7s}")
print(f"\nheld out: {HELDOUT_FEATURES}\nreward:   {REWARD_FEATURES}")

condition         d_heldout             95% CI  d_reward   Phi_A   Phi_B
peft                 0.1707 [ 0.1256, 0.2303]    0.2421   2.744   3.613
rlsf_w3_0.0          0.1802 [ 0.1301, 0.2446]    0.2410   2.769   3.598
rlsf_w3_2.0          0.2133 [ 0.1614, 0.2779]    0.2570   2.791   3.653
rlsf_w3_6.0          0.1912 [ 0.1423, 0.2544]    0.2377   2.742   3.636

held out: ['ttr', 'root_ttr', 'marker_rate']
reward:   ['lex_density', 'sent_len_mean', 'sent_len_var']


In [20]:
# Read down the omega ordering, not across the row. The Goodhart claim needs both directions in
# the same arms: judge score up, held-out distance up (further from the target register).
init = table['peft']
print(f"{'arm':16s} {'d(Phi_A)':>9s} {'d(Phi_B)':>9s} {'d(heldout)':>11s} {'d(reward)':>10s}")
for cond in ARM_CONDS:
    r = table[cond]
    da = r['phi_a'] - init['phi_a'] if None not in (r['phi_a'], init['phi_a']) else float('nan')
    db = r['phi_b'] - init['phi_b'] if None not in (r['phi_b'], init['phi_b']) else float('nan')
    print(f"{cond:16s} {da:+9.3f} {db:+9.3f} "
          f"{r['dist_heldout'] - init['dist_heldout']:+11.4f} "
          f"{r['dist_reward'] - init['dist_reward']:+10.4f}")
print('\nAgainst peft, the frozen initialization every arm started from.')

arm               d(Phi_A)  d(Phi_B)  d(heldout)  d(reward)
rlsf_w3_0.0         +0.025    -0.014     +0.0095    -0.0011
rlsf_w3_2.0         +0.047    +0.041     +0.0427    +0.0149
rlsf_w3_6.0         -0.002    +0.023     +0.0205    -0.0043

Against peft, the frozen initialization every arm started from.


In [21]:
# Which held-out feature moved, signed. A distance that grew says only that something did.
names = HELDOUT_FEATURES + REWARD_FEATURES
print(f"{'condition':16s}" + ''.join(f'{n:>14s}' for n in names))
for cond, r in table.items():
    print(f'{cond:16s}' + ''.join(f"{r['z'][n]:+14.3f}" for n in names))
print('\nSigned z against the target-register centroid; 0 is the target, sign gives the direction.')

condition                  ttr      root_ttr   marker_rate   lex_density sent_len_mean  sent_len_var
peft                    -0.023        -0.100        +0.136        -0.233        +0.040        -0.053
rlsf_w3_0.0             +0.003        -0.100        +0.150        -0.235        +0.013        -0.052
rlsf_w3_2.0             -0.011        -0.085        +0.195        -0.249        +0.030        -0.056
rlsf_w3_6.0             -0.011        -0.089        +0.169        -0.230        +0.027        -0.054

Signed z against the target-register centroid; 0 is the target, sign gives the direction.


---
## 8 — Paired bootstrap

In [22]:
BOOT = ['peft', *ARM_CONDS]
for metric in ('chrf', 'bleu', 'comet'):
    cmd = (f"{PY} manage.py bootstrap --metric {metric} --conditions {' '.join(BOOT)} "
           f"--split {SPLIT} --baseline peft --out")
    !{cmd}

wrote results/bootstrap_chrf_val.json

chrf paired bootstrap  (resamples=10000, split=val)
comparison          n     diff   ci95            p       sig
------------------------------------------------------------
rlsf_w3_0.0 - peft  1323  0.289  [0.048, 0.540]  0.0202  *  
rlsf_w3_2.0 - peft  1323  0.456  [0.206, 0.720]  0.0004  *  
rlsf_w3_6.0 - peft  1323  0.438  [0.203, 0.687]  0.0     *  

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_bleu_val.json

bleu paired bootstrap  (resamples=10000, split=val)
comparison          n     diff   ci95             p       sig
-------------------------------------------------------------
rlsf_w3_0.0 - peft  1323  0.141  [-0.078, 0.367]  0.2116     
rlsf_w3_2.0 - peft  1323  0.144  [-0.085, 0.381]  0.223      
rlsf_w3_6.0 - peft  1323  0.158  [-0.083, 0.397]  0.1974     

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_comet_val.json

comet paired bootstrap  (resamples=10000, split

In [23]:
!{PY} manage.py bootstrap --metric judge --conditions {' '.join(BOOT)} \
    --split {SPLIT} --baseline peft --out

wrote results/bootstrap_judge_val.json

judge paired bootstrap  (resamples=10000, split=val)
comparison          n     diff    ci95             p       sig
--------------------------------------------------------------
rlsf_w3_0.0 - peft  1323  0.025   [-0.005, 0.054]  0.1026     
rlsf_w3_2.0 - peft  1323  0.047   [0.017, 0.076]   0.0018  *  
rlsf_w3_6.0 - peft  1323  -0.002  [-0.029, 0.026]  0.9418     

* = 95% CI excludes 0 (difference significant at α=0.05)


In [24]:
# The second rater's scores, written beside the first rather than over them.
!{PY} manage.py bootstrap --metric judge --conditions {' '.join(BOOT)} --split {SPLIT} \
    --baseline peft --judge_tag gpt --out results/bootstrap_judge_gpt_{SPLIT}.json

wrote results/bootstrap_judge_gpt_val.json

judge paired bootstrap  (resamples=10000, split=val)
comparison          n     diff    ci95             p       sig
--------------------------------------------------------------
rlsf_w3_0.0 - peft  1293  -0.011  [-0.043, 0.021]  0.5248     
rlsf_w3_2.0 - peft  1290  0.039   [0.005, 0.073]   0.024   *  
rlsf_w3_6.0 - peft  1293  0.021   [-0.012, 0.053]  0.2126     

* = 95% CI excludes 0 (difference significant at α=0.05)


---
## 9 — Spend, for `docs/budget.md`

In [ ]:
total = 0.0
for tag, model, path in (('Phi_A', 'claude-haiku-4-5', 'results/judge_val_usage.json'),
                         ('Phi_B', 'gpt-5.6-terra', 'results/judge_gpt_val_usage.json')):
    d = json.loads(Path(path).read_text())
    s = d['session']
    total += s['cost_usd']
    print(f"| 2026-08-?? | {tag}, three RLSF arms on val | `{model}` | {s['calls']:,} | "
          f"${s['cost_usd']:.4f} | `{path}` |")
print(f"\npass total ${total:.4f} against ${AUTHORISED:.2f} authorised")

| 2026-08-?? | Phi_A, three RLSF arms on val | `claude-haiku-4-5` | 3,969 | $4.0546 | `results/judge_val_usage.json` |
| 2026-08-?? | Phi_B, three RLSF arms on val | `gpt-5.6-terra` | 3,969 | $2.6047 | `results/judge_gpt_val_usage.json` |

pass total $6.6593 against $5.74 authorised
